# 1. Facciamo la query da UniProt

Import necessary packages and modules

In [ ]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re
import pprint

: 

In [ ]:
pp = pprint.PrettyPrinter()

Define global variables to handle the connection with the UniProt website. We use the Session object to allow retries in case of temporary service unavaiability. We set 5 as tha max number of retries.

In [ ]:
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
# force the retrival to recall the api if errors 500, 502, 503, 504 happens
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

Define functions to handle API calls and pagination (useful when number of entries exceed the current limit)

In [ ]:
# Function that extract from the header the cursor/token for the next batch to be retrived and generate the url for the next request
def get_next_link(headers):
    if "Link" in headers:
        # The regular expression is used to extract the next link for pagination
        re_next_link = re.compile(r'<(.+)>; rel="next"')
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)

# This function actually retrieve the next data batch in the search.
# The function act as an iterator, yielding the next result batch at every call
# The function terminates after the last batch has been returned. In this case,
# the next link will be None
def get_batch(batch_url):
    while batch_url:
        # Run the API call
        response = session.get(batch_url)
        # Will raise an error if an error status code is obtained
        response.raise_for_status()
        # Get the total number of entries in the search
        total = response.headers["x-total-results"]
        # Yield the response and the total number of entries
        yield response, total #--> GENERATOR: obtain the data of current batch and yield them
        # Get the link to the API call for the next data batch
        batch_url = get_next_link(response.headers)

# NB: the last batch has no cursor so the last url will be empty

In [ ]:
batch_size = 500
url = "https://rest.uniprot.org/uniprotkb/search?compressed=false&format=json&query=%28%28ft_signal_exp%3A*%29+AND+%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28taxonomy_id%3A2759%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28existence%3A1%29%29&size=500"
# compressed=false --> in order to be able to manage the response
# format=json --> (NOT fasta)

## 1.2 Function for additional filter
Function that given one entry as json, perform a more advance filter. Returns true if property is soddisfied, false otherwise


In [ ]:
def filter_function(entry):
  '''This function returns true if the entry has a signal peptide that starts at the
   beginning of the chain and is longer or equal to 14 amino acids, excluding not cleaved ones'''
  for i in entry['features']:
    if i['type']=='Signal':
      if i['location']['start']['value']==1:
        if i['description']=='Not Cleaved': return False
        elif i['location']['end']['value']!='?' and i['location']['end']['value']!=None and i['location']['end']['value']>13:
          return True

## 1.3 Set the output file

In [ ]:
# We set the name of the output file, we want TSV output
output_file = "human-rev-non-frag-cc.tsv"

## 1.4 Extract fields from single entry
Fields for positive data:
- UniProt accession
- Organism name
- Eukaryotic kingdom (Metazoa, Fungi, Plants, Other)
- Protein length
- Position of the signal-peptide cleavage site

In [ ]:
def extract_fields(entry):
    #sets of filters
    UPaccession=entry['primaryAccession']
    org_name=entry['organism']['scientificName']
    euk_kingdom=entry['organism']['lineage'][1]
    seq_len=entry['sequence']['length']

    #TODO: missing transmembran helix filter, to be integrated into the pipeline

## 1.5 Retrive dataset in TSV format

Define a function to retrieve the datset and format it in TSV. It takes in input:

- a search URL corresponding the the specific API call to run (to be build using the UniProt website)
- a filter function to filter entries according to specific criteria
- and extract function used to get specific fields of interest from the JSON format
- the output file name where the final TSV will be saved

In [ ]:
def prova(search_url, filter_function):
    # If we specify a function as argument of another function means that we can call that function by using the name of the argument
    filtered_json = []
    n_total, n_filtered = 0, 0
    # Run the API call in batches
    for batch, total in get_batch(search_url):
        # parse the JSON body of the response
        batch_json = json.loads(batch.text)
        # filter the entries
        for entry in batch_json["results"]:
            n_total += 1
            # Check if the entry passes the filter (if not ignore it)
            if filter_function(entry):
                n_filtered += 1
                print(entry)

In [ ]:
def get_dataset(search_url, filter_function, extract_function, output_file_name):
    # If we specify a function as argument of another function means that we can call that function by using the name of the argument
    filtered_json = []
    n_total, n_filtered = 0, 0
    # Run the API call in batches
    for batch, total in get_batch(search_url):
        # parse the JSON body of the response
        batch_json = json.loads(batch.text)
        # filter the entries
        for entry in batch_json["results"]:
            n_total += 1
            # Check if the entry passes the filter (if not ignore it)
            if filter_function(entry):
                n_filtered += 1
                filtered_json.append(extract_function(entry))
    print(n_total, n_filtered)
    with open(output_file_name, "w") as ofs:
        for entry in filtered_json:
            # Extract the fields of interest
            #fields = extract_function(entry)
            # Print the fields in TSV format
            print(*entry, sep="\t", file=ofs)
        ofs.close

We call the above function to obtaine our dataset

In [ ]:
get_dataset(url, filter_function, extract_fields, output_file)

## TEST (to be eliminated)

In [ ]:
def prova(search_url, filter_peptide_length):
    # If we specify a function as argument of another function means that we can call that function by using the name of the argument
    filtered_json = []
    n_total, n_filtered = 0, 0
    # Run the API call in batches
    for batch, total in get_batch(search_url):
        # parse the JSON body of the response
        batch_json = json.loads(batch.text)
        # filter the entries
        for entry in batch_json["results"]:
            n_total += 1
            # Check if the entry passes the filter (if not ignore it)
            if filter_peptide_length(entry):
                n_filtered += 1
                print(entry)

In [ ]:
prova(url, filter_function)